aggregation

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import min, max, avg, sum, count
cust_df=spark.read.option("inferSchema", True).option("header", False).csv("/Volumes/izwe48catalog/we48db/testvolume/custs")
cust_df.count() # this will return the integer
# Here converting c0,c1 etc to cust id , first name etc
cust_df = cust_df.toDF("cust_id", "first_name", "last_name", "age", "prof")
cust_prof_count=cust_df.groupBy("prof").count()
cust_prof_count.show() # this will return the dataframe
# age wise count
cust_age_count=cust_df.groupBy("age").count()
cust_age_count.show()                                                                                                      
#min age by profession
cust_prof_min_count=cust_df.groupBy("prof").min("age")
cust_prof_min_count.show()
#max age by profession
cust_prof_max_count=cust_df.groupBy("prof").max("age")
cust_prof_max_count.show()
#avg age by profession
cust_prof_avg_count=cust_df.groupBy("prof").avg("age")
cust_prof_avg_count.show()
#min, max
cust_prof_min_max_count=cust_df.groupBy("prof").agg(min("age"), max("age")) # using agg function we can display multiple columns
cust_prof_min_max_count.show()
#min. max ,agg , age
cust_prof_agg_count=cust_df.groupBy("prof").agg(min("age").alias("min_age"), max("age").alias("max_age"), count("*").alias("tot_rec"))
cust_prof_agg_count.show()
cust_prof_agg_count.filter("tot_rec>200").show()

In [0]:
ctran_df = spark.read.csv("/Volumes/izwe48catalog/we48db/staging/ctrans_id.csv", header=True, inferSchema=True)
cust_df = spark.read.csv("/Volumes/izwe48catalog/we48db/staging/customer.csv", header=True, inferSchema=True)
cust_df.show()
ctran_df.show()
#left outer join/left join 
cust_tran_df = cust_df.join(ctran_df,on="cid",how="left")
cust_tran_df.show()

cust_tran1_df = cust_df.join(ctran_df,ctran_df.cid==cust_df.cid,how="left")
cust_tran1_df.show()

cust_tran1_df = cust_df.join(ctran_df.alias a,on=(cust_df.cid==a.cid),how="left")
cust_tran1_df.show()



cust_tran_df = cust_df.join(ctran_df,on="cid",how="right")
cust_tran_df.show()

In [0]:
emp_data =[(100,"john",21),(101,"prem",41),(102,"abc",71),(103,"xyz",91),(105,"sync",41)]
city_data=[(100,"chennai"),(101,"Bglr"),(102,"Pune"),(104,"Trichy")]

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

df=spark.range(5).withColumn("name1",lit("Data1"))
df1=spark.range(101,105).withColumn("name2",lit("Data2"))
df.join(df1).show()
df.crossJoin(df1).show()
df.union(df1).show()


#How to generate seq number - (sk - surrogate key) 
#if source is sending key 
#windowing functions



In [0]:
from pyspark.sql.functions import monotonically_increasing_id
emp_data =[(100,"john",21),(101,"prem",41),(102,"abc",71),(103,"xyz",91),(105,"sync",41)]
emp_data_df=spark.createDataFrame(emp_data,["id","name","age"])
#emp_sk_data = emp_data_df.withColumn("cust_sk",monotonically_increasing_id())
emp_sk_data_df = emp_data_df.repartition(2).withColumn("cust_sk",monotonically_increasing_id())
emp_sk_data_df.show()

#window - group
##window two things we can mention - combination of partition and order by 
#partition 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
sales_data =[("Arun","Sales",31,10000),
("Meera","Sales",45,20000),
("Irfan","Engg",29,30000),
("Raj","Mktg",52,28000),
("Ram","Engg",27,27000),
("Sneha","Mktg",21,35000),
("Raja","Sales",30,23000),
("Satish","Mktg",26,23000),
("Naga","Sales",31,10000),
("Priya","Engg",29,30000),
("Ravi","Sales",34,89000)]

sales_df=spark.createDataFrame(sales_data,["Name","Dept","Age","Salary"])
sales_df.show()
w_spec=Window.partitionBy("Dept").orderBy("Salary","Age")
sales_df.withColumn("row_number",F.row_number().over(w_spec)).withColumn("rank",F.rank().over(w_spec)).withColumn("dense_rank",F.dense_rank().over(w_spec)).show()

#window + aggregation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
sales_data =[("Arun","Sales",31,10000),
("Meera","Sales",45,20000),
("Irfan","Engg",29,30000),
("Raj","Mktg",52,28000),
("Ram","Engg",27,27000),
("Sneha","Mktg",21,35000),
("Raja","Sales",30,23000),
("Satish","Mktg",26,23000),
("Naga","Sales",31,10000),
("Priya","Engg",29,30000),
("Ravi","Sales",34,89000)]

sales_df=spark.createDataFrame(sales_data, ["Name","Dept","Age","Salary"])
sales_df.show()
#sales_df.groupBy("Dept").show()
w_spec=Window.partitionBy("Dept")

sales_agg_df=sales_df.withColumn("min_salary",F.min("Salary").over(w_spec)).withColumn("max_salary",F.max("Salary").over(w_spec)).withColumn("avg_salary",F.avg("Salary").over(w_spec))
sales_agg_df.show()


In [0]:

# ranking function 

# syntax 

# row_number().over(Window.partitionBy("column").orderBy("column"))
# rank().over(Window.partitionBy("column").orderBy("column"))
# dense_rank().over(Window.partitionBy("column").orderBy("column"))

from pyspark.sql.functions import *

from pyspark.sql.window import Window

w_spec=Window.partitionBy("dept").orderBy("salary")



df.withColumn("r_num",row_number().over(w_spec)).withColumn("rank",rank().over(w_spec)).withColumn("d_rank",dense_rank().over(w_spec)).show()